In [1]:
import numpy as np
import pandas as pd
import seaborn as sb
import matplotlib.pyplot as plt
sb.set()

from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

In [2]:
cardiodata = pd.read_csv('cardio_train.csv', sep=';')
cardiodata.head()

,id,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio
0,0,18393,2,168,62.0,110,80,1,1,0,0,1,0
1,1,20228,1,156,85.0,140,90,3,1,0,0,1,1
2,2,18857,1,165,64.0,130,70,3,1,0,0,0,1
3,3,17623,2,169,82.0,150,100,1,1,0,0,1,1
4,4,17474,1,156,56.0,100,60,1,1,0,0,0,0


In [3]:
# 1.inspect and clean cardiovascular dataset
print("Original shape:", cardiodata.shape)
print(cardiodata.columns.tolist())
cardiodata.head()

Original shape: (70000, 13)
['id', 'age', 'gender', 'height', 'weight', 'ap_hi', 'ap_lo', 'cholesterol', 'gluc', 'smoke', 'alco', 'active', 'cardio']


,id,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio
0,0,18393,2,168,62.0,110,80,1,1,0,0,1,0
1,1,20228,1,156,85.0,140,90,3,1,0,0,1,1
2,2,18857,1,165,64.0,130,70,3,1,0,0,0,1
3,3,17623,2,169,82.0,150,100,1,1,0,0,1,1
4,4,17474,1,156,56.0,100,60,1,1,0,0,0,0


In [4]:
cardiodata.info()
print(cardiodata.describe(include='all').T)

# Check missing values
print("Missing values per column:\n", cardiodata.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 70000 entries, 0 to 69999
Data columns (total 13 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   id           70000 non-null  int64  
 1   age          70000 non-null  int64  
 2   gender       70000 non-null  int64  
 3   height       70000 non-null  int64  
 4   weight       70000 non-null  float64
 5   ap_hi        70000 non-null  int64  
 6   ap_lo        70000 non-null  int64  
 7   cholesterol  70000 non-null  int64  
 8   gluc         70000 non-null  int64  
 9   smoke        70000 non-null  int64  
 10  alco         70000 non-null  int64  
 11  active       70000 non-null  int64  
 12  cardio       70000 non-null  int64  
dtypes: float64(1), int64(12)
memory usage: 6.9 MB
               count          mean           std      min       25%      50%  \
id           70000.0  49972.419900  28851.302323      0.0  25006.75  50001.5   
age          70000.0  19468.865814   2467.251667  

In [10]:
#Remove rows where systolic <= diastolic (ap_hi <= ap_lo)
before = cardiodata.shape[0]
cardiodata = cardiodata[cardiodata['ap_hi'] > cardiodata['ap_lo']]
print(f"Removed {before - cardiodata.shape[0]} rows with ap_hi <= ap_lo")

#Remove extreme heights and weights
cardiodata = cardiodata[(cardiodata['height'] >= 100) & (cardiodata['height'] <= 240)]
cardiodata = cardiodata[(cardiodata['weight'] >= 30) & (cardiodata['weight'] <= 250)]
print("After removing implausible height/weight rows:", cardiodata.shape)

#Drop any duplicates
before = cardiodata.shape[0]
cardiodata = cardiodata.drop_duplicates()
print("Removed duplicates:", before - cardiodata.shape[0])

#Drop all NA rows
before = cardiodata.shape[0]
cardiodata = cardiodata.dropna()
print("Removed NA rows:", before - cardiodata.shape[0])

Removed 1236 rows with ap_hi <= ap_lo
After removing implausible height/weight rows: (68730, 13)
Removed duplicates: 0
Removed NA rows: 0


In [5]:
#2. Convert age to years if value>150
if cardiodata['age'].max() > 150:
    cardiodata['age_years'] = (cardiodata['age'] / 365).round().astype(int)
else:
    # if value<=150
    cardiodata['age_years'] = cardiodata['age'].astype(int)
    
print("Age (years) - min, max:", cardiodata['age_years'].min(), cardiodata['age_years'].max())

Age (years) - min, max: 30 65


In [15]:
#3. Compute BMI
# BMI = weight (kg) / (height (m))^2
cardiodata['height_m'] = cardiodata['height'] / 100.0
cardiodata['BMI'] = cardiodata['weight'] / (cardiodata['height_m'] ** 2)

print(cardiodata['BMI'].describe())

count    70000.000000
mean        27.556513
std          6.091511
min          3.471784
25%         23.875115
50%         26.374068
75%         30.222222
max        298.666667
Name: BMI, dtype: float64


In [16]:
# 4)Engineer risk features
# BMI risk: 1 if BMI >= 30 (obese), else 0 
cardiodata['bmi_risk'] = (cardiodata['BMI'] >= 30).astype(int)

# High blood pressure: systolic >= 140 OR diastolic >= 90 (1 = high BP)
cardiodata['high_bp'] = ((cardiodata['ap_hi'] >= 140) | (cardiodata['ap_lo'] >= 90)).astype(int)

# Cholesterol risk: dataset usually codes 1=normal, 2=above norm, 3=well above
# Mark 2 or 3 as risk
def cholesterol_risk(x):
    if x > 1:  
        return 1
    else:
        return 0
cardiodata['chol_risk'] = cardiodata['cholesterol'].apply(cholesterol_risk)

# Glucose risk: same as cholesterol risk
def glucose_risk(x):
    if x > 1:  
        return 1
    else:
        return 0
cardiodata['gluc_risk'] = cardiodata['gluc'].apply(glucose_risk)

# Lifestyle binary flags (smoke, alco, active) typically already 0/1 in dataset
# We define: smoke_risk: 1, alco_risk: 1, inactive_risk: 1 (active==1 => no risk)
cardiodata['smoke_risk'] = cardiodata['smoke'].astype(int)
cardiodata['alco_risk'] = cardiodata['alco'].astype(int)
cardiodata['inactive_risk'] = (cardiodata['active'] == 0).astype(int)

# check how many people are in each risk category
risk_cols = ['age_risk','bmi_risk','chol_risk','gluc_risk','smoke_risk','alco_risk','inactive_risk']
print(cardiodata[risk_cols].sum())

age_risk         15900
bmi_risk         18474
chol_risk        17615
gluc_risk        10521
smoke_risk        6169
alco_risk         3764
inactive_risk    13739
dtype: int64


In [17]:
# 5)Calculate Lifestyle Risk Index (LRI) on 0-7 scale.The higher the value (0–7), the higher the overall risk.
cardiodata['LRI'] = (
    cardiodata['bmi_risk'] +
    cardiodata['high_bp'] +
    cardiodata['cholesterol'] +  # already encoded as 1 or 2 for elevated levels
    cardiodata['gluc']           # already encoded as 1 or 2 for elevated levels
)
print("Lifestyle Risk Index (LRI) created. Summary:")
print(cardiodata['LRI'].describe())

Lifestyle Risk Index (LRI) created. Summary:
count    70000.000000
mean         3.210171
std          1.404116
min          2.000000
25%          2.000000
50%          3.000000
75%          4.000000
max          8.000000
Name: LRI, dtype: float64
